# P28 SGN clustering

Set the paths marked in the code before running this notebook.

Reproduces the accepted result from `P28_2.ipynb` using the original raw count files. Use the same analysis environment.

Only branches that determine cell retention are retained: three global HVG branches for the residual immune-cell union, then 2000 / 2000 / 3000 / 2000 HVGs for target extraction, local screening, marker-supported cell selection, and final clustering. All numerical parameters, final plotting code, 127 marker genes and established colors follow the accepted notebook.

Displays compact QC figures, two final UMAPs and three final DotPlots. Writes one canonical raw-count H5AD under the configured directory; an existing matching file is validated and retained. Cross-dataset gene harmonization remains pending.

In [ ]:
#1. =========== Load raw counts and set the random seed
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
from datetime import datetime, timezone
import os, gzip, random, tempfile, hashlib, importlib.metadata, json, warnings

RANDOM_SEED, N_THREADS = (42, 1)
ROOT = Path(".")  # Set the project root before running this template.
STAGE, DATASET_ID = ("P28", "GSE202920")
# Set the temporary working directory or file path.
cache_root = ROOT
cache_root.mkdir(parents=True, exist_ok=True)
runtime = tempfile.TemporaryDirectory(prefix="runtime_", dir=cache_root)
os.environ["NUMBA_CACHE_DIR"] = runtime.name
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[key] = str(N_THREADS)
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy import sparse
from scipy.io import mmread
from sklearn.mixture import GaussianMixture
from threadpoolctl import threadpool_limits

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
thread_guard = threadpool_limits(limits=N_THREADS)
sc.settings.n_jobs, sc.settings.verbosity = (N_THREADS, 0)
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 10,
        "figure.dpi": 120,
        "figure.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
SOURCES = {"GSM6138444": "GSM6138444_p28_1", "GSM6138445": "GSM6138445_p28_2"}
# Set the original count-data file or directory path.
DATA_DIR = ROOT
raw_by_library = {}
for library, prefix in SOURCES.items():
    # Set the data file or directory path.
    paths = {
        kind: ROOT
        for kind, suffix in (("matrix", "mtx.gz"), ("features", "tsv.gz"), ("barcodes", "tsv.gz"))
    }
    features = pd.read_csv(paths["features"], sep="\t", header=None)
    barcodes = pd.read_csv(paths["barcodes"], sep="\t", header=None)[0].astype(str)
    with gzip.open(paths["matrix"], "rb") as handle:
        matrix = mmread(handle).T.tocsr()
    matrix.eliminate_zeros()
    assert matrix.shape == (len(barcodes), len(features)) and barcodes.is_unique
    assert (
        np.isfinite(matrix.data).all()
        and (matrix.data >= 0).all()
        and np.equal(matrix.data, np.floor(matrix.data)).all()
    )
    if features.shape[1] > 2:
        assert features.iloc[:, 2].eq("Gene Expression").all()
    symbols = features.iloc[:, 1].astype(str)
    codes, unique = pd.factorize(symbols, sort=False)
    mapping = sparse.csr_matrix(
        (np.ones(len(codes), dtype=np.int64), (np.arange(len(codes)), codes)), shape=(len(codes), len(unique))
    )
    X = (matrix @ mapping).tocsr()
    assert int(X.sum()) == int(matrix.sum())
    var = pd.DataFrame(index=pd.Index(unique, name="gene_symbol"))
    var["source_gene_ids"] = features.iloc[:, 0].astype(str).groupby(codes).agg(";".join).to_numpy()
    var["mt"] = var.index.str.match("(?i)^mt[-_]")
    var["hb"] = var.index.str.match("(?i)^hb[ab]-")
    var["ribo"] = var.index.str.match("^Rp[sl]\\d")
    var["technical"] = var.index.str.match("(?i)^ercc[-_]\\d+$|^(e?gfp\\d*|tdtomato|mcherry|wpre)$")
    obs = pd.DataFrame(
        index=pd.Index([f"{DATASET_ID}__{library}__{b}" for b in barcodes], name="cell_id_global")
    )
    obs["original_cell_id"] = barcodes.to_numpy()
    obs["dataset_id"], obs["library_id"], obs["sample_id"] = (DATASET_ID, library, library)
    obs["biological_replicate_id"], obs["strain"] = (library + "_pooled", "C57/B6")
    obs["pooled_mice_per_protocol"], obs["genotype"], obs["sex"] = (7, "not_reported", "not_reported")
    obs["technology"], obs["stage_original"], obs["stage_standardized"] = ("10x_3prime_v3.1", "P28", STAGE)
    obs["source_file"] = str(paths["matrix"])
    a = ad.AnnData(X=X, obs=obs, var=var)
    a.layers["counts"] = X.copy()
    raw_by_library[library] = a


In [ ]:
#2. =========== Library QC and count-preserving merge
BLOOD_MARKERS = ["Ptprc", "Lyz2", "Gypa"]
NEURON_MARKERS = ["Snap25", "Tubb3", "Stmn2", "Elavl3", "Rbfox3"]
SGN_MARKERS = ["Isl1", "Gata3", "Mafb", "Nefl", "Nefm"]
EXCLUSION = {
    "Immune": ["Ptprc", "Lyz2", "Tyrobp", "C1qa"],
    "Glia": ["Sox10", "Mpz", "Plp1", "Mbp"],
    "Hair": ["Myo7a", "Pou4f3", "Otof"],
    "Epithelial": ["Epcam", "Krt8", "Krt18"],
    "Vascular": ["Pecam1", "Kdr", "Esam"],
    "Mesenchymal": ["Col1a1", "Col1a2", "Dcn"],
}


def gene_counts(a, genes):
    selected = a.var_names.isin(genes)
    return np.asarray(a.layers["counts"][:, selected].sum(axis=1)).ravel()


for library, a in raw_by_library.items():
    X = a.layers["counts"]
    total = np.asarray(X.sum(axis=1)).ravel()
    a.obs["total_counts"], a.obs["n_genes"] = (total, X.getnnz(axis=1))
    for feature in ("mt", "hb", "ribo", "technical"):
        a.obs[f"pct_{feature}"] = (
            100 * np.asarray(X[:, a.var[feature].to_numpy()].sum(axis=1)).ravel() / np.maximum(total, 1)
            if a.var[feature].any()
            else np.nan
        )
    a.obs["endogenous_counts"] = np.asarray(X[:, ~a.var["technical"].to_numpy()].sum(axis=1)).ravel()
    a.obs["blood_marker_positive"] = gene_counts(a, BLOOD_MARKERS) > 0
    a.obs["neuronal_genes_detected"] = X[:, a.var_names.isin(NEURON_MARKERS)].getnnz(axis=1)
    a.obs["log_counts"] = np.log10(1 + total)
    a.obs["log_genes"] = np.log10(1 + a.obs["n_genes"])
MT_MAX_PCT = {"GSM6138444": 30, "GSM6138445": 30}
post_mito = {}
for library, a in raw_by_library.items():
    o = a.obs
    o["mito_or_zero_fail"] = (o.pct_mt > MT_MAX_PCT[library]) | (o.endogenous_counts == 0) | (o.n_genes == 0)
    post_mito[library] = a[~o.mito_or_zero_fail.to_numpy()].copy()
    b = post_mito[library]
HB_MAX_PCT = {"GSM6138444": 3, "GSM6138445": 3}
post_blood, post_blood_gmm, qc_ledger = ({}, {}, [])
for library, b in post_mito.items():
    strong = b.obs.blood_marker_positive.to_numpy()
    hb = b.obs.pct_hb.to_numpy() >= HB_MAX_PCT[library]
    c = b[~(strong | hb)].copy()
    post_blood[library] = c
    z = c.obs[["log_counts", "log_genes"]].to_numpy()
    post_blood_gmm[library] = GaussianMixture(2, n_init=10, random_state=RANDOM_SEED).fit(z)
    source = raw_by_library[library].obs
    source["exclusion_reason"] = ""
    source.loc[source.mito_or_zero_fail, "exclusion_reason"] = "high_mito_or_zero"
    source.loc[b.obs_names[strong | hb], "exclusion_reason"] = "blood"
    qc_ledger.append({"library_id": library, "raw": len(source), "post_mito": b.n_obs, "post_blood": c.n_obs})
LOW_GENES, LOW_COUNTS, LOW_POSTERIOR = (250, 750, 0.99)
SCRUBLET_PCS = 30
pre_doublet, doublet_diagnostics = ({}, {})
for row in qc_ledger:
    library = row["library_id"]
    b = post_blood[library]
    z = b.obs[["log_counts", "log_genes"]].to_numpy()
    model = post_blood_gmm[library]
    posterior = model.predict_proba(z)[:, np.argmin(model.means_[:, 0])]
    hits = np.max(
        [
            b.X[:, b.var_names.isin(genes)].getnnz(axis=1)
            for genes in [NEURON_MARKERS, SGN_MARKERS, *EXCLUSION.values()]
        ],
        axis=0,
    )
    low_fail = (
        (posterior > LOW_POSTERIOR)
        & (b.obs.n_genes.to_numpy() < LOW_GENES)
        & (b.obs.total_counts.to_numpy() < LOW_COUNTS)
        & (hits < 2)
    )
    raw_by_library[library].obs.loc[b.obs_names[low_fail], "exclusion_reason"] = "low_complexity"
    c = b[~low_fail].copy()
    pre_doublet[library] = c
    row["cell_called"] = c.n_obs
    rate = float(np.clip(0.008 * raw_by_library[library].n_obs / 1000, 0.01, 0.15))
    eligible = ~(c.var.mt | c.var.hb | c.var.technical).to_numpy()
    probe = c[:, eligible].copy()
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="invalid value encountered in scalar divide",
            category=RuntimeWarning,
            module="scanpy\\.preprocessing\\._scrublet\\.core",
        )
        sc.pp.scrublet(
            probe,
            expected_doublet_rate=rate,
            n_prin_comps=SCRUBLET_PCS,
            use_approx_neighbors=False,
            random_state=RANDOM_SEED,
            threshold=1,
            verbose=False,
        )
    c.obs["doublet_score"] = probe.obs.doublet_score
    observed = probe.obs.doublet_score.to_numpy()
    simulated = probe.uns["scrublet"]["doublet_scores_sim"]
    doublet_diagnostics[library] = {"expected_rate": rate, "observed": observed, "simulated": simulated}
DOUBLET_THRESHOLDS = {"GSM6138444": 0.3, "GSM6138445": 0.4}
qc_by_library = {}
for row in qc_ledger:
    library = row["library_id"]
    b = pre_doublet[library]
    o = b.obs
    o["doublet_candidate"] = o.doublet_score >= DOUBLET_THRESHOLDS[library]
    o["doublet_fail"] = o.doublet_candidate
    o["doublet_status"] = np.where(o.doublet_fail, "predicted_doublet", "below_review_threshold")
    source = raw_by_library[library].obs
    source.loc[o.index[o.doublet_fail], "exclusion_reason"] = "predicted_doublet"
    source["qc_pass"] = source.exclusion_reason.eq("")
    c = b[~o.doublet_fail.to_numpy()].copy()
    c.X = c.layers["counts"].copy()
    qc_by_library[library] = c
    c.uns["doublet_parameters"] = {
        "expected_rate": doublet_diagnostics[library]["expected_rate"],
        "threshold": DOUBLET_THRESHOLDS[library],
        "seed": RANDOM_SEED,
    }
    row["final_QC"] = c.n_obs
qc_summary = pd.DataFrame(qc_ledger).set_index("library_id")
reference = next(iter(qc_by_library.values())).var
for c in qc_by_library.values():
    assert c.var.equals(reference)
    assert c.obs_names.is_unique and (c.X != c.layers["counts"]).nnz == 0
qc_full = ad.concat(list(qc_by_library.values()), join="outer", merge="same", index_unique=None)
assert qc_full.n_obs == int(qc_summary.final_QC.sum()) and qc_full.obs_names.is_unique
assert int(qc_full.X.sum()) == sum((int(c.X.sum()) for c in qc_by_library.values()))
assert (qc_full.X != qc_full.layers["counts"]).nnz == 0
assert (
    np.isfinite(qc_full.X.data).all()
    and (qc_full.X.data >= 0).all()
    and np.equal(qc_full.X.data, np.floor(qc_full.X.data)).all()
)


In [ ]:
#3. =========== Residual immune removal and SGN region
HVG_COUNTS = (2000, 3000, 5000)
GLOBAL_PCS, GLOBAL_NEIGHBORS, GLOBAL_RESOLUTION, MIN_DIST = (50, 30, 0.7, 0.35)
MIN_GENE_CELLS, NORMALIZE_TARGET, HVG_SPAN = (5, 10000, 0.3)


def prepare_expression(raw, min_cells):
    usable = (raw.layers["counts"].getnnz(axis=0) >= min_cells) & ~(
        raw.var.mt | raw.var.hb | raw.var.technical
    ).to_numpy()
    a = raw[:, usable].copy()
    a.X = a.layers["counts"].astype(np.float32).copy()
    sc.pp.normalize_total(a, target_sum=NORMALIZE_TARGET)
    sc.pp.log1p(a)
    return a


def run_branches(expression, pcs, neighbors, resolution, hvg_counts):
    branches = {}
    for n_hvg in hvg_counts:
        assert expression.n_vars >= n_hvg and expression.n_obs > pcs
        a = expression.copy()
        batch_key = "library_id" if a.obs.library_id.nunique() > 1 else None
        sc.pp.highly_variable_genes(
            a, n_top_genes=n_hvg, flavor="seurat_v3", layer="counts", batch_key=batch_key, span=HVG_SPAN
        )
        h = a[:, a.var.highly_variable].copy()
        h.X = h.X.toarray()
        sc.pp.scale(h, max_value=10)
        sc.tl.pca(h, n_comps=pcs, svd_solver="arpack", random_state=RANDOM_SEED)
        a.obsm["X_pca"] = h.obsm["X_pca"].copy()
        a.uns["pca"] = h.uns["pca"].copy()
        sc.pp.neighbors(
            a,
            n_neighbors=neighbors,
            n_pcs=pcs,
            transformer="sklearn",
            metric="euclidean",
            random_state=RANDOM_SEED,
        )
        sc.tl.leiden(
            a,
            resolution=resolution,
            flavor="igraph",
            n_iterations=2,
            directed=False,
            random_state=RANDOM_SEED,
            key_added="cluster",
        )
        a.uns["analysis_parameters"] = {
            "n_hvg": n_hvg,
            "pcs": pcs,
            "neighbors": neighbors,
            "resolution": resolution,
            "seed": RANDOM_SEED,
            "batch_correction": "none",
        }
        branches[n_hvg] = a
    return branches


def marker_values(a, genes):
    genes = list(dict.fromkeys(genes))
    present = [g for g in genes if g in qc_full.var_names]
    raw = qc_full[a.obs_names, present].layers["counts"].toarray()
    denominator = np.asarray(a.layers["counts"].sum(axis=1)).ravel()
    return (present, raw, np.log1p(raw / np.maximum(denominator[:, None], 1) * NORMALIZE_TARGET))


def marker_table(a, genes):
    genes, raw, log = marker_values(a, genes)
    groups = a.obs.cluster.astype(str)
    avg = pd.DataFrame(log, index=a.obs_names, columns=genes).groupby(groups, observed=True).mean()
    pct = pd.DataFrame(raw > 0, index=a.obs_names, columns=genes).groupby(groups, observed=True).mean() * 100
    return (avg, pct)


def lineage_table(a):
    rows = []
    for cluster in a.obs.cluster.cat.categories:
        b = qc_full[a.obs_names[a.obs.cluster == cluster]]
        row = {
            "cluster": cluster,
            "cells": b.n_obs,
            "neuron_2gene_fraction": np.mean(b.X[:, b.var_names.isin(NEURON_MARKERS)].getnnz(axis=1) >= 2),
            "SGN_2gene_fraction": np.mean(b.X[:, b.var_names.isin(SGN_MARKERS)].getnnz(axis=1) >= 2),
        }
        for name, genes in EXCLUSION.items():
            row[name + "_2gene_fraction"] = np.mean(b.X[:, b.var_names.isin(genes)].getnnz(axis=1) >= 2)
        rows.append(row)
    return pd.DataFrame(rows).set_index("cluster")


global_expression = prepare_expression(qc_full, MIN_GENE_CELLS)
global_branches = run_branches(global_expression, GLOBAL_PCS, GLOBAL_NEIGHBORS, GLOBAL_RESOLUTION, HVG_COUNTS)
IMMUNE_REVIEW = [
    "Tyrobp",
    "C1qa",
    "C1qb",
    "C1qc",
    "Aif1",
    "Ctss",
    "Csf1r",
    "Fcer1g",
    "Laptm5",
    "Ptprc",
    "Lyz2",
]
IMMUNE_PCT_MIN, IMMUNE_GENES_MIN, IMMUNE_MEAN_MIN, NEURON_MAX = (20, 5, 0.5, 0.1)
immune_core = [g for g in IMMUNE_REVIEW if g not in BLOOD_MARKERS]
immune_ids = set()
for n, a in global_branches.items():
    avg, pct = marker_table(a, immune_core)
    evidence = lineage_table(a)
    selected = avg.index[
        ((pct >= IMMUNE_PCT_MIN).sum(axis=1) >= IMMUNE_GENES_MIN)
        & (avg.mean(axis=1) >= IMMUNE_MEAN_MIN)
        & (evidence.neuron_2gene_fraction < NEURON_MAX)
    ]
    ids = set(a.obs_names[a.obs.cluster.isin(selected)])
    immune_ids |= ids
qc_summary["post_doublet"] = qc_summary.final_QC
for library, c in qc_by_library.items():
    ids = c.obs_names[c.obs_names.isin(immune_ids)]
    source = raw_by_library[library].obs
    source.loc[ids, "exclusion_reason"] = "residual_immune_lineage"
    source["qc_pass"] = source.exclusion_reason.eq("")
    qc_by_library[library] = c[~c.obs_names.isin(immune_ids)].copy()
    qc_summary.loc[library, "final_QC"] = qc_by_library[library].n_obs
qc_summary = qc_summary[["raw", "post_mito", "post_blood", "cell_called", "post_doublet", "final_QC"]]
qc_full = ad.concat(list(qc_by_library.values()), join="outer", merge="same", index_unique=None)
assert qc_full.n_obs == int(qc_summary.final_QC.sum()) and (qc_full.X != qc_full.layers["counts"]).nnz == 0
assert int(qc_full.X.sum()) == sum((int(c.X.sum()) for c in qc_by_library.values()))
del global_branches, global_expression
global_expression = prepare_expression(qc_full, MIN_GENE_CELLS)
global_branches = run_branches(global_expression, GLOBAL_PCS, GLOBAL_NEIGHBORS, GLOBAL_RESOLUTION, (2000,))
GLOBAL_HVG, NEURON_FRACTION, SGN_FRACTION = (2000, 0.5, 0.5)
HAIR_MAX, GLIA_MAX, IMMUNE_MAX = (0.1, 0.3, 0.05)
region_ids = {}
for n, a in global_branches.items():
    evidence = lineage_table(a)
    selected = evidence.index[
        (evidence.neuron_2gene_fraction >= NEURON_FRACTION)
        & (evidence.SGN_2gene_fraction >= SGN_FRACTION)
        & (evidence.Hair_2gene_fraction < HAIR_MAX)
        & (evidence.Glia_2gene_fraction < GLIA_MAX)
        & (evidence.Immune_2gene_fraction < IMMUNE_MAX)
    ]
    region_ids[n] = set(a.obs_names[a.obs.cluster.isin(selected)])
global_adata = global_branches[GLOBAL_HVG]
target_ids = global_adata.obs_names[global_adata.obs_names.isin(region_ids[GLOBAL_HVG])]
target_raw = qc_full[target_ids].copy()
target_raw.X = target_raw.layers["counts"].copy()
target_raw.obs["global_cluster"] = global_adata.obs.loc[target_ids, "cluster"].astype(str)
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), layout="constrained")
for ax, library in zip(axes[:2], SOURCES):
    diag = doublet_diagnostics[library]
    bins = np.linspace(0, max(diag["observed"].max(), diag["simulated"].max()), 50)
    ax.hist(diag["simulated"], bins=bins, density=True, alpha=0.35, color="#AE814C", label="Simulated")
    ax.hist(diag["observed"], bins=bins, density=True, histtype="step", color="#527A9D", label="Observed")
    ax.axvline(DOUBLET_THRESHOLDS[library], color="0.25", linestyle="--")
    ax.set(xlabel="Scrublet score", ylabel="Density", title=library)
axes[0].legend(frameon=False, fontsize=7)
for library in qc_summary.index:
    axes[2].plot(range(len(qc_summary.columns)), qc_summary.loc[library], "o-", label=library)
axes[2].set(
    xticks=range(len(qc_summary.columns)),
    xticklabels=["Raw", "Mito", "Blood", "Complexity", "Doublets", "Immune"],
    ylabel="Cells",
    title="QC retention",
)
axes[2].tick_params(axis="x", rotation=35, labelsize=7)
axes[2].legend(frameon=False, fontsize=7)
plt.show()
plt.close(fig)


In [ ]:
#4. =========== Joint target QC and local HVG2000 clustering
MIN_PANEL_HITS = 2
MYELIN_GENES = ["Mpz", "Mbp", "Plp1"]
QC_COLORS = {"GSM6138444": "#557B9F", "GSM6138445": "#B7864A"}
assert target_raw.obs_names.equals(target_ids), "Run through #3 before target QC."
assert (target_raw.X != target_raw.layers["counts"]).nnz == 0
assert (target_raw.X != qc_full[target_ids, target_raw.var_names].layers["counts"]).nnz == 0
target_region_raw = target_raw.copy()
q = target_region_raw.obs[
    ["library_id", "global_cluster", "total_counts", "n_genes", "pct_mt", "pct_ribo", "doublet_score"]
].copy()
x = target_region_raw.layers["counts"]
assert np.allclose(np.asarray(x.sum(axis=1)).ravel(), q.total_counts)
assert (q.total_counts > 0).all()
panels = {"Neuron": NEURON_MARKERS, "SGN": SGN_MARKERS, "Myelin": MYELIN_GENES, "Immune": IMMUNE_REVIEW}
for name, genes in panels.items():
    mask = target_region_raw.var_names.isin(genes)
    assert mask.sum() >= 2, f"Too few available genes in {name}."
    q[name + "_hits"] = np.asarray((x[:, mask] > 0).sum(axis=1)).ravel()
    q[name + "_UMI"] = np.asarray(x[:, mask].sum(axis=1)).ravel()
neural_genes = list(dict.fromkeys(NEURON_MARKERS + SGN_MARKERS))
q["neural_UMI"] = gene_counts(target_region_raw, neural_genes)
q["Sox10_UMI"] = gene_counts(target_region_raw, ["Sox10"])
q["pct_neural"] = 100 * q.neural_UMI / q.total_counts
q["pct_myelin"] = 100 * q.Myelin_UMI / q.total_counts
q["neural_support"] = (q.Neuron_hits >= MIN_PANEL_HITS) & (q.SGN_hits >= MIN_PANEL_HITS)
q["myelin_detected"] = q.Myelin_hits >= MIN_PANEL_HITS
q["sox10_detected"] = q.Sox10_UMI > 0
q["immune_detected"] = q.Immune_hits >= MIN_PANEL_HITS
q["myelin_sox10"] = q.myelin_detected & q.sox10_detected
q["evidence_group"] = np.select(
    [q.neural_support & q.myelin_detected, q.neural_support, q.myelin_detected],
    ["Neural + myelin", "Neural support", "Myelin; limited neural"],
    default="Limited neural",
)
q["log_counts"], q["log_genes"] = (np.log10(1 + q.total_counts), np.log10(1 + q.n_genes))
target_qc_cells = q
libraries = list(q.library_id.unique())
fig, axes = plt.subplots(1, 4, figsize=(11.2, 3.1), layout="constrained")
metrics = {
    "log_counts": "log10(1 + UMI)",
    "log_genes": "log10(1 + genes)",
    "pct_mt": "Mito UMI (%)",
    "doublet_score": "Scrublet score",
}
for ax, (key, title) in zip(axes, metrics.items()):
    for j, library in enumerate(libraries):
        values = q.loc[q.library_id.eq(library), key].to_numpy()
        if len(values) > 1 and np.ptp(values) > 0:
            v = ax.violinplot(values, positions=[j], widths=0.65, showmedians=True, showextrema=False)
            v["bodies"][0].set(facecolor=QC_COLORS[library], edgecolor="none", alpha=0.7)
            v["cmedians"].set_color("0.25")
        else:
            ax.scatter([j], [values[0]], c=QC_COLORS[library], s=10)
    ax.set(xticks=range(len(libraries)), xticklabels=libraries, title=title)
    ax.tick_params(axis="x", labelrotation=25, labelsize=7)
fig.suptitle("P28 target candidates: quality by library")
plt.show()
plt.close(fig)
LOCAL_LOW_GENES, LOCAL_MT_MAX = (1000, 10)
LOCAL_PCS, LOCAL_NEIGHBORS, LOCAL_RESOLUTION, LOCAL_MIN_CELLS = (25, 15, 0.5, 3)
assert target_region_raw.obs_names.equals(target_ids), "Run the preceding target QC setup."
assert target_qc_cells.index.equals(target_region_raw.obs_names)
target_qc_audit = target_qc_cells.copy()
target_qc_audit["low_genes"] = target_qc_audit.n_genes < LOCAL_LOW_GENES
target_qc_audit["high_mt"] = target_qc_audit.pct_mt > LOCAL_MT_MAX
target_qc_audit["target_qc_pass"] = ~(target_qc_audit.low_genes & target_qc_audit.high_mt)
target_qc_audit["target_exclusion_reason"] = np.where(
    target_qc_audit.target_qc_pass, "", "low_genes_and_high_mito"
)
target_qc_audit["target_qc_status"] = np.where(
    target_qc_audit.target_qc_pass, "Retained", "Excluded: local QC"
)
fig, axes = plt.subplots(
    1, len(libraries), figsize=(9, 3.4), sharex=True, sharey=True, squeeze=False, layout="constrained"
)
for ax, library in zip(axes.ravel(), libraries):
    b = target_qc_audit.loc[target_qc_audit.library_id.eq(library)]
    for status, color in (("Retained", "#557B9F"), ("Excluded: local QC", "#BB767E")):
        d = b.loc[b.target_qc_status.eq(status)]
        ax.scatter(
            d.log_genes, d.pct_mt, c=color, s=9, alpha=0.65, linewidths=0, label=f"{status} ({len(d)})"
        )
    ax.axvline(np.log10(1 + LOCAL_LOW_GENES), color="0.4", ls="--", lw=0.8)
    ax.axhline(LOCAL_MT_MAX, color="0.4", ls="--", lw=0.8)
    ax.set(title=library, xlabel="log10(1 + detected genes)", ylabel="Mito UMI (%)")
    ax.legend(frameon=False, fontsize=7)
fig.suptitle("Joint QC: low genes AND high mitochondrial fraction")
plt.show()
keep_ids = target_qc_audit.index[target_qc_audit.target_qc_pass]
target_raw = target_region_raw[keep_ids].copy()
target_raw.X = target_raw.layers["counts"].copy()
for key in (
    "target_qc_pass",
    "target_exclusion_reason",
    "neural_support",
    "myelin_detected",
    "myelin_sox10",
    "immune_detected",
    "pct_neural",
    "pct_myelin",
    "Sox10_UMI",
):
    target_raw.obs[key] = target_qc_audit.loc[keep_ids, key]
target_raw.uns["target_qc_parameters"] = {
    "low_genes": LOCAL_LOW_GENES,
    "high_mt_pct": LOCAL_MT_MAX,
    "logic": "AND",
    "input_cells": target_region_raw.n_obs,
}
assert target_raw.n_obs + int((~target_qc_audit.target_qc_pass).sum()) == target_region_raw.n_obs
assert target_raw.obs.library_id.nunique() == target_region_raw.obs.library_id.nunique()
assert target_raw.n_obs > max(LOCAL_PCS, LOCAL_NEIGHBORS + 1)
assert (target_raw.X != target_region_raw[keep_ids].layers["counts"]).nnz == 0
local_expression = prepare_expression(target_raw, LOCAL_MIN_CELLS)
local_branches = {}
for n_hvg in (2000,):
    assert local_expression.n_vars >= n_hvg
    a = local_expression.copy()
    sc.pp.highly_variable_genes(
        a, n_top_genes=n_hvg, flavor="seurat_v3", layer="counts", batch_key="library_id", span=HVG_SPAN
    )
    hvg = a.var.highly_variable.to_numpy()
    h = a[:, hvg].copy()
    h.X = h.X.toarray()
    sc.pp.scale(h, max_value=10)
    sc.tl.pca(h, n_comps=LOCAL_PCS, svd_solver="arpack", random_state=RANDOM_SEED)
    a.obsm["X_pca"], a.uns["pca"] = (h.obsm["X_pca"].copy(), h.uns["pca"].copy())
    sc.pp.neighbors(
        a,
        use_rep="X_pca",
        n_neighbors=LOCAL_NEIGHBORS,
        n_pcs=LOCAL_PCS,
        transformer="sklearn",
        metric="euclidean",
        random_state=RANDOM_SEED,
    )
    sc.tl.leiden(
        a,
        resolution=LOCAL_RESOLUTION,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=RANDOM_SEED,
        key_added="cluster",
    )
    a.uns["analysis_parameters"] = {
        "n_hvg": n_hvg,
        "pcs": LOCAL_PCS,
        "neighbors": LOCAL_NEIGHBORS,
        "resolution": LOCAL_RESOLUTION,
        "seed": RANDOM_SEED,
        "batch_key": "library_id",
        "batch_correction": "none",
    }
    assert a.obs_names.equals(target_raw.obs_names)
    assert not a.var[["mt", "hb", "technical"]].to_numpy().any()
    assert (a.layers["counts"] != target_raw[:, a.var_names].layers["counts"]).nnz == 0
    local_branches[n_hvg] = a
    del h


In [ ]:
#5. =========== Subtype candidate selection and HVG3000 clustering
REVIEW_MARKER_PANELS = {
    "Observed": ["Apoe", "Apod", "Ptgds", "Gjc3", "Sostdc1"],
    "lncRNA": ["Miat", "Meg3", "Snhg11", "Malat1"],
    "Glia / myelin": ["Sox10", "Plp1", "Mpz", "Mbp"],
    "SGC auxiliary": ["Glul", "Kcnj10", "Fabp7", "Gata2"],
    "Schwann": ["Ncmap", "Prx", "Pmp22", "Egr2"],
    "Immune": ["Ptprc", "Lyz2", "Tyrobp", "C1qa", "Csf1r", "Ctss"],
    "Hair cells": ["Myo7a", "Pou4f3", "Otof", "Slc17a8", "Slc26a5"],
    "Stroma": ["Col1a1", "Col1a2", "Dcn"],
    "Vascular": ["Pecam1", "Kdr", "Esam"],
    "Epithelium": ["Epcam", "Krt8", "Krt18"],
}
group_colors = {
    "Neuron / SGN": "#557B9F",
    "Ia": "#B7864A",
    "Ib": "#9A6E9C",
    "Ic": "#6E9368",
    "II": "#BB767E",
    "Glia / myelin": "#608F8A",
    "Immune": "#9F795D",
    "Ib/Ic shared": "#8680AC",
    "II auxiliary": "#A8A05D",
    "Ib candidates": "#8195A4",
    "Observed": "#BD927D",
    "lncRNA": "#849F7B",
    "SGC auxiliary": "#8C8A59",
    "Schwann": "#B3789D",
    "Hair cells": "#6F99B2",
    "Stroma": "#C09C62",
    "Vascular": "#926B77",
    "Epithelium": "#759B96",
}
reference = local_branches[2000]
SGC_AUX_GENES = REVIEW_MARKER_PANELS["SGC auxiliary"]
genes, raw, _ = marker_values(reference, ["Sox10"] + SGC_AUX_GENES)
raw_df = pd.DataFrame(raw, index=reference.obs_names, columns=genes)
detected = raw_df.gt(0)
sgc2 = detected[SGC_AUX_GENES].sum(axis=1).ge(2)
expanded_codetection = pd.DataFrame({"SGC2_Sox10": sgc2 & detected.Sox10})
P28_TYPES = ["Type Ia", "Type Ib", "Type Ic", "Type II"]
P28_CLUSTER_MAP = {
    2000: {"3": "Type Ia", "2": "Type Ib", "1": "Type Ic", "6": "Type II"},
    3000: {"2": "Type Ia", "1": "Type Ib", "4": "Type Ic", "6": "Type II"},
    5000: {"2": "Type Ia", "1": "Type Ib", "4": "Type Ic", "6": "Type II"},
}
screen = local_branches[2000]
assert screen.n_obs == 588, "Use the local HVG2000 result from #4."
assert screen.obs_names.equals(target_raw.obs_names)
mixed_ids = screen.obs_names[screen.obs.cluster.astype(str).eq("5")]
seed = screen.obs.cluster.astype(str).map(P28_CLUSTER_MAP[2000]).fillna("Unresolved")
candidate = seed.isin(P28_TYPES)
assert candidate.sum() == 362
glial = expanded_codetection.loc[screen.obs_names, "SGC2_Sox10"] | screen.obs.myelin_sox10.astype(bool)
neural = screen.obs.neural_support.astype(bool)
keep = candidate & neural & ~glial
p28_selection = target_qc_audit.copy()
p28_selection["p28_seed_type"] = seed.reindex(p28_selection.index).fillna("Unresolved")
p28_selection["p28_keep"] = keep.reindex(p28_selection.index, fill_value=False)
p28_selection["p28_selection_reason"] = "Unresolved subtype"
p28_selection.loc[~p28_selection.target_qc_pass, "p28_selection_reason"] = "Prior target QC"
p28_selection.loc[screen.obs_names[~neural], "p28_selection_reason"] = "Limited neuronal evidence"
p28_selection.loc[screen.obs_names[glial], "p28_selection_reason"] = "Glial co-detection"
p28_selection.loc[mixed_ids, "p28_selection_reason"] = "Neuron-glia mixed cluster"
p28_selection.loc[p28_selection.p28_keep, "p28_selection_reason"] = "Retained subtype candidate"
p28_raw = target_raw[screen.obs_names[keep]].copy()
p28_raw.X = p28_raw.layers["counts"].copy()
p28_raw.obs["p28_seed_type"] = pd.Categorical(seed.loc[p28_raw.obs_names], categories=P28_TYPES, ordered=True)
p28_raw.uns["selection_parameters"] = {
    "source": "Reviewed HVG2000 subtype candidate clusters",
    "exclude": "Sox10 with >=2 myelin genes or >=2 SGC auxiliary genes",
}
assert p28_raw.obs.p28_seed_type.value_counts().reindex(P28_TYPES, fill_value=0).gt(0).all()
assert p28_raw.obs.library_id.nunique() == 2
assert (p28_raw.layers["counts"] != target_raw[p28_raw.obs_names].layers["counts"]).nnz == 0
p28_expression = prepare_expression(p28_raw, LOCAL_MIN_CELLS)
p28_branches = {}
for n_hvg in (3000,):
    a = p28_expression.copy()
    assert a.n_vars >= n_hvg
    assert a.n_obs > max(LOCAL_PCS, LOCAL_NEIGHBORS + 1)
    sc.pp.highly_variable_genes(
        a, n_top_genes=n_hvg, flavor="seurat_v3", layer="counts", batch_key="library_id", span=HVG_SPAN
    )
    hvg = a.var.highly_variable.to_numpy()
    h = a[:, hvg].copy()
    h.X = h.X.toarray()
    sc.pp.scale(h, max_value=10)
    sc.tl.pca(h, n_comps=LOCAL_PCS, svd_solver="arpack", random_state=RANDOM_SEED)
    a.obsm["X_pca"] = h.obsm["X_pca"].copy()
    a.uns["pca"] = h.uns["pca"].copy()
    sc.pp.neighbors(
        a,
        use_rep="X_pca",
        n_neighbors=LOCAL_NEIGHBORS,
        n_pcs=LOCAL_PCS,
        transformer="sklearn",
        metric="euclidean",
        random_state=RANDOM_SEED,
    )
    sc.tl.leiden(
        a,
        resolution=LOCAL_RESOLUTION,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=RANDOM_SEED,
        key_added="cluster",
    )
    a.uns["analysis_parameters"] = {
        "n_hvg": n_hvg,
        "pcs": LOCAL_PCS,
        "neighbors": LOCAL_NEIGHBORS,
        "resolution": LOCAL_RESOLUTION,
        "seed": RANDOM_SEED,
        "batch_key": "library_id",
        "batch_correction": "none",
    }
    assert a.obs_names.equals(p28_raw.obs_names)
    assert not a.var[["mt", "hb", "technical"]].to_numpy().any()
    assert (a.layers["counts"] != p28_raw[:, a.var_names].layers["counts"]).nnz == 0
    p28_branches[n_hvg] = a
    del h


In [ ]:
#6. =========== Marker-supported cells and final HVG2000 clustering
CORE = {
    "Type Ia": ["Calb2", "Rxrg", "Pcdh20", "B3gat1", "Trim54", "Cacna1b"],
    "Type Ib": ["Calb1", "Ttn"],
    "Type Ic": ["Lypd1", "Grm8", "Pou4f1", "Kcnc2"],
    "Type II": ["Prph", "Ngfr", "Th", "Cux2", "Npy", "Plk5", "Cilp", "Grik3", "Grin2c", "Kcnc4"],
}
IB_SHARED = ["Ntng1", "Runx1", "Nalcn", "Ncald"]
MIN_CORE_GENES, MIN_CORE_UMI, MIN_ANCHORS = (2, 3, 3)
MARGIN_MIN, LOG_SD_FLOOR, Z_CAP = (0.1, 0.25, 5.0)
CURRENT_TYPES = {"1": "Type Ia", "0": "Type Ib", "2": "Type Ic", "3": "Type II"}
base = p28_branches[3000]
assert base.n_obs == 358 and base.obs_names.equals(p28_raw.obs_names)
assert set(base.obs.cluster.astype(str)) == {"0", "1", "2", "3", "4"}
source_type = base.obs.cluster.astype(str).map(CURRENT_TYPES).fillna("Unresolved")
eligible = source_type.ne("Unresolved")
assert eligible.sum() == 335
feature_genes = list(dict.fromkeys([g for gs in CORE.values() for g in gs] + IB_SHARED))
found, counts, expression = marker_values(base, feature_genes)
assert set(found) == set(feature_genes), "Required subtype markers are missing."
counts = pd.DataFrame(counts, index=base.obs_names, columns=found)
expression = pd.DataFrame(expression, index=base.obs_names, columns=found)
p28_core_hits = pd.DataFrame({t: counts[gs].gt(0).sum(axis=1) for t, gs in CORE.items()})
p28_core_umi = pd.DataFrame({t: counts[gs].sum(axis=1) for t, gs in CORE.items()})
strict_support = p28_core_hits.ge(MIN_CORE_GENES) & p28_core_umi.ge(MIN_CORE_UMI)
cell_support = strict_support.copy()
cell_support["Type Ib"] |= counts[CORE["Type Ib"]].max(axis=1).ge(2) & counts[IB_SHARED].gt(0).sum(axis=1).ge(
    2
)
anchors = pd.DataFrame({t: eligible & source_type.eq(t) & strict_support[t] for t in P28_TYPES})
anchor_counts = anchors.groupby(base.obs.library_id, observed=True).sum().astype(int)
assert anchor_counts.shape[0] == 2
assert (
    anchor_counts.ge(MIN_ANCHORS).all().all()
), "Too few opposite-library anchors; inspect the reference counts."
p28_cell_audit = base.obs[["library_id", "n_genes", "total_counts", "pct_mt", "doublet_score"]].copy()
p28_cell_audit["source_cluster_HVG3000"] = base.obs.cluster.astype(str)
p28_cell_audit["source_type"] = source_type
p28_cell_audit["profile_type"] = "Not evaluated"
p28_cell_audit["runner_up_type"] = "Not evaluated"
p28_cell_audit["marker_supported"] = False
p28_cell_audit["best_distance"] = np.nan
p28_cell_audit["distance_margin"] = np.nan
type_names = np.array(P28_TYPES)
for library in base.obs.library_id.unique():
    query_ids = base.obs_names[eligible & base.obs.library_id.eq(library)]
    reference_mask = base.obs.library_id.ne(library)
    blocks = [expression.loc[anchors[t] & reference_mask].to_numpy() for t in P28_TYPES]
    assert all((len(x) >= MIN_ANCHORS for x in blocks))
    center = np.stack([x.mean(axis=0) for x in blocks]).mean(axis=0)
    variance = np.stack([((x - center) ** 2).mean(axis=0) for x in blocks]).mean(axis=0)
    scale = np.maximum(np.sqrt(variance), LOG_SD_FLOOR)
    profiles = np.stack([x.mean(axis=0) for x in blocks])
    prototypes = np.clip((profiles - center) / scale, -Z_CAP, Z_CAP)
    query = np.clip((expression.loc[query_ids].to_numpy() - center) / scale, -Z_CAP, Z_CAP)
    distances = np.sqrt(((query[:, None, :] - prototypes[None, :, :]) ** 2).mean(axis=2))
    ranked = np.argsort(distances, axis=1)
    best, second = (ranked[:, 0], ranked[:, 1])
    rows = np.arange(len(query_ids))
    d1, d2 = (distances[rows, best], distances[rows, second])
    margin = (d2 - d1) / np.maximum(d2 + d1, 1e-08)
    supported = cell_support.loc[query_ids, P28_TYPES].to_numpy()[rows, best]
    p28_cell_audit.loc[query_ids, "profile_type"] = type_names[best]
    p28_cell_audit.loc[query_ids, "runner_up_type"] = type_names[second]
    p28_cell_audit.loc[query_ids, "marker_supported"] = supported
    p28_cell_audit.loc[query_ids, "best_distance"] = d1
    p28_cell_audit.loc[query_ids, "distance_margin"] = margin
p28_cell_audit["retain"] = (
    eligible & p28_cell_audit.marker_supported & p28_cell_audit.distance_margin.ge(MARGIN_MIN)
)
p28_cell_audit["decision"] = np.select(
    [~eligible, ~p28_cell_audit.marker_supported, p28_cell_audit.distance_margin.lt(MARGIN_MIN)],
    ["Weak subtype cluster", "Insufficient marker evidence", "Ambiguous subtype"],
    default="Retained candidate",
)
p28_cell_audit["candidate_type"] = np.where(p28_cell_audit.retain, p28_cell_audit.profile_type, "Unresolved")
p28_cell_audit["changed_type"] = p28_cell_audit.retain & p28_cell_audit.profile_type.ne(source_type)
assert np.isfinite(p28_cell_audit.loc[eligible, "distance_margin"]).all()
p28_clean_raw = p28_raw[p28_cell_audit.index[p28_cell_audit.retain]].copy()
for obj in (p28_clean_raw,):
    for key in (
        "source_cluster_HVG3000",
        "source_type",
        "profile_type",
        "runner_up_type",
        "marker_supported",
        "distance_margin",
        "candidate_type",
        "decision",
    ):
        obj.obs[key] = p28_cell_audit.loc[obj.obs_names, key]
p28_clean_raw.obs["candidate_type"] = pd.Categorical(
    p28_clean_raw.obs.candidate_type, categories=P28_TYPES, ordered=True
)
p28_clean_raw.uns["cell_type_screen"] = {
    "core_genes": CORE,
    "Ib_shared": IB_SHARED,
    "min_core_genes": MIN_CORE_GENES,
    "min_core_umi": MIN_CORE_UMI,
    "min_opposite_library_anchors": MIN_ANCHORS,
    "margin_min": MARGIN_MIN,
    "log_sd_floor": LOG_SD_FLOOR,
    "z_cap": Z_CAP,
    "method": "Opposite-library class-balanced mean marker profiles",
}
p28_final_selection = p28_selection.copy()
p28_final_selection["stage21_decision"] = p28_cell_audit.decision.reindex(p28_final_selection.index).fillna(
    "Not in stage21 input"
)
p28_final_selection["final_candidate_keep"] = p28_final_selection.index.isin(p28_clean_raw.obs_names)
remaining = pd.crosstab(p28_clean_raw.obs.candidate_type, p28_clean_raw.obs.library_id).reindex(
    index=P28_TYPES, columns=base.obs.library_id.unique(), fill_value=0
)
assert remaining.ge(2).all().all(), "Too few retained cells in a type-library group; inspect the audit."
assert p28_clean_raw.n_obs == int(p28_cell_audit.retain.sum())
assert (p28_clean_raw.layers["counts"] != p28_raw[p28_clean_raw.obs_names].layers["counts"]).nnz == 0
assert p28_clean_raw.n_obs > max(
    LOCAL_PCS + 1, LOCAL_NEIGHBORS + 1
), "Too few retained cells for the current graph settings."
p28_clean_expression = prepare_expression(p28_clean_raw, LOCAL_MIN_CELLS)
p28_clean_branches = run_branches(p28_clean_expression, LOCAL_PCS, LOCAL_NEIGHBORS, LOCAL_RESOLUTION, (2000,))
for a in p28_clean_branches.values():
    assert a.obs_names.equals(p28_clean_raw.obs_names)
    assert not a.var[["mt", "hb", "technical"]].to_numpy().any()
    assert (a.layers["counts"] != p28_clean_raw[:, a.var_names].layers["counts"]).nnz == 0
sc.tl.umap(p28_clean_branches[2000], min_dist=MIN_DIST, random_state=RANDOM_SEED)


In [ ]:
#7. =========== Final annotation, UMAPs and 127-marker DotPlots
def expanded_dotplot(stats, panels, title):
    kept = {name: [g for g in gs if g in stats["pct"].columns] for name, gs in panels.items()}
    genes = [g for gs in kept.values() for g in gs]
    pct, scaled = (stats["pct"][genes], stats["scaled"][genes])
    COLUMN_GAP_RATIO, DMAX_PT = (0.5, 6)
    pitch = DMAX_PT * (1 + COLUMN_GAP_RATIO) / 72
    width, height = (max(7, len(genes) * pitch + 3.4), max(3.9, len(pct) * 0.27 + 2.9))
    left, bottom, matrix_w, matrix_h = (2.1, 0.95, len(genes) * pitch, max(0.6, len(pct) * 0.27))
    fig = plt.figure(figsize=(width, height), facecolor="white")
    ax = fig.add_axes([left / width, bottom / height, matrix_w / width, matrix_h / height])
    ax.set_facecolor("white")
    xx, yy = np.meshgrid(np.arange(len(genes)), np.arange(len(pct)))
    cmap = LinearSegmentedColormap.from_list(
        "expanded_purple", ["#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F"]
    )
    dots = ax.scatter(
        xx.ravel(),
        yy.ravel(),
        s=(pct.to_numpy() / 100 * DMAX_PT**2).ravel(),
        c=scaled.to_numpy().ravel(),
        cmap=cmap,
        vmin=-1,
        vmax=1,
        linewidths=0,
    )
    ax.set(
        xlim=(-0.5, len(genes) - 0.5),
        ylim=(len(pct) - 0.5, -0.5),
        xticks=np.arange(len(genes)),
        yticks=np.arange(len(pct)),
    )
    ax.set_xticklabels(genes, rotation=60, ha="right", fontsize=7)
    ax.set_yticklabels([f"{x} ({stats['sizes'].loc[x]})" for x in pct.index], fontsize=7)
    ax.tick_params(length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    offset = 0
    for name, gs in kept.items():
        if not gs:
            continue
        ax.plot(
            [offset - 0.4, offset + len(gs) - 0.6],
            [-0.85, -0.85],
            color=group_colors[name],
            lw=3,
            clip_on=False,
        )
        ax.text(
            offset + (len(gs) - 1) / 2,
            -1.05,
            name,
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=35,
            clip_on=False,
        )
        offset += len(gs)
    cax = fig.add_axes([left / width, (bottom + matrix_h + 1.1) / height, 1.35 / width, 0.1 / height])
    cb = fig.colorbar(dots, cax=cax, orientation="horizontal", ticks=[-1, -0.5, 0, 0.5, 1])
    cb.outline.set_visible(False)
    cax.set_title("Avg. Exp. (gene z-score)", fontsize=7, pad=4)
    cax.tick_params(labelsize=6, length=2)
    handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            color="0.45",
            markeredgewidth=0,
            markersize=DMAX_PT * np.sqrt(p / 100),
            label=str(p),
        )
        for p in (0, 25, 50, 75, 100)
    ]
    fig.legend(
        handles=handles,
        title="% Exp.",
        ncol=5,
        loc="upper right",
        bbox_to_anchor=(0.98, 0.97),
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        handletextpad=0.25,
        columnspacing=0.5,
    )
    fig.text(0.02, 0.98, title, ha="left", va="top", fontsize=10)
    fig.canvas.draw()
    gap = (ax.transData.transform((1, 0))[0] - ax.transData.transform((0, 0))[0]) / (
        DMAX_PT * fig.dpi / 72
    ) - 1
    assert abs(gap - COLUMN_GAP_RATIO) < 0.05
    plt.show()
    plt.close(fig)


FINAL_HVG = 2000
FINAL_CLUSTER_MAP = {"1": "Type Ia", "0": "Type Ib", "2": "Type Ic", "3": "Type II"}
P28_COLORS = {"Type Ia": "#557B9F", "Type Ib": "#B7864A", "Type Ic": "#9A6E9C", "Type II": "#6E9368"}
POINT_DIAMETER_SCALE = 2.5
POINT_SIZE = 1.35 * POINT_DIAMETER_SCALE**2
POINT_ALPHA, LABEL_SIZE = (0.82, 6.75)
p28_final = p28_clean_branches[FINAL_HVG].copy()
clusters = p28_final.obs.cluster.astype(str)
assert p28_final.n_obs == 284 and set(clusters) == set(FINAL_CLUSTER_MAP)
p28_final.obs["cell_type"] = pd.Categorical(
    clusters.map(FINAL_CLUSTER_MAP), categories=P28_TYPES, ordered=True
)
p28_final.obs["annotation_disagreement"] = p28_final.obs.cell_type.astype(
    str
) != p28_final.obs.candidate_type.astype(str)
p28_final.uns["cell_type_colors"] = [P28_COLORS[t] for t in P28_TYPES]
p28_final.uns["cell_type_palette"] = P28_COLORS.copy()
p28_final.uns["final_annotation"] = {
    "source": "Reviewed HVG2000 clusters",
    "cluster_map": FINAL_CLUSTER_MAP.copy(),
}
final_sizes = p28_final.obs.cell_type.value_counts().reindex(P28_TYPES)
assert final_sizes.tolist() == [52, 93, 117, 22]
assert np.array_equal(p28_final.obsm["X_umap"], p28_clean_branches[FINAL_HVG].obsm["X_umap"])
xy = p28_final.obsm["X_umap"]
labels = p28_final.obs.cell_type.astype(str).to_numpy()
draw_order = np.random.default_rng(RANDOM_SEED).permutation(p28_final.n_obs)
point_colors = np.array([P28_COLORS[t] for t in labels])
margin = np.ptp(xy, axis=0) * 0.09
lower, upper = (xy.min(axis=0) - margin, xy.max(axis=0) + margin)
for mode in ("legend", "label"):
    fig, ax = plt.subplots(figsize=(6.6 if mode == "legend" else 4.4, 4.1), facecolor="white")
    if mode == "legend":
        fig.subplots_adjust(right=0.6)
    ax.set_facecolor("white")
    ax.scatter(*xy[draw_order].T, c=point_colors[draw_order], s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0)
    ax.set(
        xlabel="UMAP1",
        ylabel="UMAP2",
        title="P28 SGN types / HVG2000",
        xlim=(lower[0], upper[0]),
        ylim=(lower[1], upper[1]),
        box_aspect=1,
    )
    ax.grid(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.1)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if mode == "legend":
        handles = [
            Line2D(
                [],
                [],
                marker="o",
                linestyle="",
                color=P28_COLORS[t],
                markeredgewidth=0,
                markersize=5,
                label=f"{t} ({final_sizes[t]})",
            )
            for t in P28_TYPES
        ]
        ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=7)
    else:
        texts = []
        for t in P28_TYPES:
            points = xy[labels == t]
            anchor = points[np.argmin(np.sum((points - np.median(points, axis=0)) ** 2, axis=1))]
            texts.append(
                ax.text(
                    *anchor,
                    t,
                    ha="center",
                    va="center",
                    fontsize=LABEL_SIZE,
                    color="black",
                    path_effects=[pe.withStroke(linewidth=1.45, foreground="white")],
                )
            )
        fig.canvas.draw()
        for i, txt in enumerate(texts):
            for previous in texts[:i]:
                renderer = fig.canvas.get_renderer()
                if txt.get_window_extent(renderer).overlaps(previous.get_window_extent(renderer)):
                    x, y = txt.get_position()
                    txt.set_position((x, y + np.ptp(xy[:, 1]) * 0.06))
                    fig.canvas.draw()
    fig.canvas.draw()
    box = ax.get_window_extent()
    assert abs(box.width - box.height) < 1
    plt.show()
    plt.close(fig)
P28_FINAL_MAIN_PANELS = {
    "Ia": [
        "Calb2",
        "Rxrg",
        "Pcdh20",
        "Trim54",
        "Cacna1b",
        "B3gat1",
        "Mdga1",
        "Tmod4",
        "Afap1l1",
        "Btbd9",
        "Maf",
    ],
    "Ib": ["Calb1", "Ttn", "Kcnd2", "Pls3", "Plcb4", "Kcnh2", "Edil3"],
    "Ic": ["Lypd1", "Grm8", "Pou4f1", "Kcnc2", "Sphkap", "Cd164l2", "Rgs12", "Pcsk2", "Tmem117", "Prkar2b"],
    "II": [
        "Prph",
        "Ngfr",
        "Th",
        "Calca",
        "Cux2",
        "Npy",
        "Plk5",
        "Cilp",
        "Grik3",
        "Grin2c",
        "Kcnc4",
        "Cacna1g",
        "Piezo2",
        "Grin3a",
    ],
}
P28_FINAL_AUX_PANELS = {
    "Neuron / SGN": [
        "Snap25",
        "Tubb3",
        "Stmn2",
        "Elavl3",
        "Rbfox3",
        "Isl1",
        "Gata3",
        "Mafb",
        "Nefl",
        "Nefm",
        "Nefh",
    ],
    "Type I shared": ["Prox1", "Epha4", "Trpm2", "Grm7", "Slc17a7"],
    "Ib/Ic shared": [
        "Runx1",
        "Ntng1",
        "Nalcn",
        "Ncald",
        "Kcnip2",
        "Kcnj9",
        "Dcc",
        "Slit2",
        "Sema3a",
        "Cplx2",
        "Slc17a6",
    ],
    "Ia channels": ["Cacna1h", "Cacna2d1", "Kcnq4", "Chrna1", "Chrna4"],
    "Ic gradient": ["Drd1", "Grik4", "Scn2b"],
    "II receptors": ["Htr2c", "Htr7", "Chrna7", "Drd5"],
    "Ib candidates": ["Kctd8", "Efr3a", "Pdzd2", "Sema3e"],
    "Regional / channel": ["Lrrc52"],
}
group_colors.update(
    {
        "Type I shared": "#608F8A",
        "Ia channels": group_colors["Ia"],
        "Ic gradient": group_colors["Ic"],
        "II receptors": group_colors["II"],
        "Regional / channel": "#8195A4",
    }
)
final_panels = {**P28_FINAL_MAIN_PANELS, **P28_FINAL_AUX_PANELS, **REVIEW_MARKER_PANELS}
requested = [g for gs in final_panels.values() for g in gs]
assert len(requested) == len(set(requested)), "Duplicate genes in the marker panels."
genes, raw, log = marker_values(p28_final, requested)
labels = p28_final.obs.cell_type
log_frame = pd.DataFrame(log, index=p28_final.obs_names, columns=genes)
detected_frame = pd.DataFrame(raw > 0, index=p28_final.obs_names, columns=genes)
avg = log_frame.groupby(labels, observed=True).mean().reindex(P28_TYPES)
pct = detected_frame.groupby(labels, observed=True).mean().reindex(P28_TYPES) * 100
scaled = (avg - avg.mean()) / avg.std(ddof=0).clip(lower=1e-08)
assert np.isfinite(scaled.to_numpy()).all()
assert pct.ge(0).all().all() and pct.le(100).all().all()
p28_final_marker_stats = {"mean_log": avg, "pct": pct, "scaled": scaled, "sizes": final_sizes.copy()}
expanded_dotplot(p28_final_marker_stats, P28_FINAL_MAIN_PANELS, "P28 SGN types / subtype-enriched genes")
expanded_dotplot(
    p28_final_marker_stats, P28_FINAL_AUX_PANELS, "P28 SGN types / shared and functional markers"
)
expanded_dotplot(p28_final_marker_stats, REVIEW_MARKER_PANELS, "P28 SGN types / other lineage markers")


In [ ]:
#8. =========== Canonical raw-count H5AD export and read-back validation
# Set the result file path; uses P28_GSE202920__GSM6138444-GSM6138445__SGN_for_integration.h5ad generated by P28 clustering export.
export_path = (
    ROOT
)
export_ids = p28_final.obs_names.copy()
export_genes = global_expression.var_names.copy()
assert export_ids.is_unique and export_genes.is_unique
assert export_genes.str.casefold().is_unique
assert export_ids.equals(p28_clean_raw.obs_names)
assert set(export_ids) == set(p28_final_selection.index[p28_final_selection.final_candidate_keep])
assert set(export_ids) == set(p28_cell_audit.index[p28_cell_audit.retain])
assert global_expression.obs_names.equals(qc_full.obs_names)
assert all((a.var_names.equals(export_genes) for a in global_branches.values()))
mother_counts = qc_full.layers["counts"]
feature_keep = (mother_counts.getnnz(axis=0) >= MIN_GENE_CELLS) & ~qc_full.var[["mt", "hb", "technical"]].any(
    axis=1
).to_numpy()
assert qc_full.var_names[feature_keep].equals(export_genes)
assert not export_ids.isin(immune_ids).any()
export_X = qc_full[export_ids, export_genes].layers["counts"].tocsr(copy=True)
export_X.sum_duplicates()
export_X.eliminate_zeros()
export_X.sort_indices()
assert np.isfinite(export_X.data).all()
assert (export_X.data >= 0).all()
assert np.equal(export_X.data, np.floor(export_X.data)).all()
assert export_X.data.max(initial=0) <= np.iinfo(np.int32).max
export_X = export_X.astype(np.int32)
assert (export_X != p28_clean_raw[:, export_genes].layers["counts"]).nnz == 0
shared_genes = export_genes.intersection(p28_final.var_names, sort=False)
assert (
    export_X[:, export_genes.get_indexer(shared_genes)] != p28_final[:, shared_genes].layers["counts"]
).nnz == 0
export_obs = p28_final.obs.drop(columns=[c for c in p28_final.obs if c.startswith("program_")]).copy()
for table in (p28_final_selection, p28_cell_audit):
    missing_columns = table.columns.difference(export_obs.columns)
    export_obs = export_obs.join(table.loc[export_ids, missing_columns])
export_obs["cell_id_global"] = export_ids.astype(str)
export_obs["source_cell_id"] = export_ids.astype(str)
export_obs["study_id"] = export_obs.dataset_id.astype(str)
export_obs["sample_id_original"] = export_obs.sample_id.astype(str)
batch = export_obs.dataset_id.astype(str) + "__" + export_obs.library_id.astype(str)
for key in ("sample_id", "batch_id_global", "integration_batch"):
    export_obs[key] = batch
for key in ("stage_analysis", "stage_standardized"):
    export_obs[key] = "P28"
export_obs["batch_stage_confounded"] = True
export_obs["selected_for_integration"] = True
export_obs["candidate_identity"] = "P28_SGN"
export_obs["annotation_status"] = "final_cluster_annotation"
export_obs["candidate_source"] = "P28-clustering.ipynb:#3->#4->#5->#6->#7"
for n, branch in p28_clean_branches.items():
    export_obs[f"final_cluster_HVG{n}"] = branch.obs.loc[export_ids, "cluster"].astype(str)
    export_obs[f"seed_cluster_HVG{n}"] = local_branches[n].obs.loc[export_ids, "cluster"].astype(str)
final_labels = export_obs.cell_type.astype(str)
disagreement = final_labels.ne(export_obs.candidate_type.astype(str))
assert disagreement.equals(export_obs.annotation_disagreement)
assert final_labels.equals(export_obs.cluster.astype(str).map(FINAL_CLUSTER_MAP))
export_obs["scanvi_label"] = pd.Categorical(
    final_labels.mask(disagreement, "Unknown"), categories=P28_TYPES + ["Unknown"]
)
export_obs["scanvi_supervision_status"] = np.where(
    disagreement, "unlabeled_annotation_disagreement", "labeled_consistent_annotation"
)
full_selected = qc_full[export_ids]
full_counts = full_selected.layers["counts"].tocsr()
export_obs["total_counts_source"] = np.asarray(full_counts.sum(axis=1)).ravel()
export_obs["n_genes_source"] = full_counts.getnnz(axis=1)
assert np.array_equal(export_obs.total_counts_source.to_numpy(), export_obs.total_counts.to_numpy())
assert np.array_equal(export_obs.n_genes_source.to_numpy(), export_obs.n_genes.to_numpy())
export_obs["n_features_source"] = np.nan
endogenous = ~full_selected.var.technical.to_numpy()
export_obs["total_counts_endogenous"] = np.asarray(full_counts[:, endogenous].sum(axis=1)).ravel()
export_obs["n_genes_endogenous"] = full_counts[:, endogenous].getnnz(axis=1)
export_obs["total_counts_analysis"] = np.asarray(export_X.sum(axis=1)).ravel()
export_obs["n_genes_analysis"] = export_X.getnnz(axis=1)
export_obs["pct_mito"] = export_obs.pct_mt
reporter = full_selected.var_names.str.match("(?i)^(e?gfp\\d*|tdtomato|mcherry|wpre)$")
export_obs["pct_reporter"] = (
    100
    * np.asarray(full_counts[:, reporter].sum(axis=1)).ravel()
    / np.maximum(export_obs.total_counts_source.to_numpy(), 1)
)
export_obs["technical_qc_pass"] = ~export_obs.mito_or_zero_fail.astype(bool)
export_obs["blood_qc_pass"] = (
    ~export_obs.blood_marker_positive.astype(bool)
    & export_obs.pct_hb.lt(export_obs.library_id.astype(str).map(HB_MAX_PCT))
    & ~export_ids.isin(immune_ids)
)
export_obs["blood_lineage_status"] = "retained_non_blood"
export_obs["predicted_doublet"] = export_obs.doublet_fail.astype(bool)
export_obs["doublet_qc_pass"] = ~export_obs.predicted_doublet
export_obs["cell_calling_qc_pass"] = export_ids.isin(
    pd.Index(np.concatenate([a.obs_names.to_numpy() for a in pre_doublet.values()]))
)
export_obs["qc_complexity_mode"] = "post_blood_two_component_GMM"
for library, model in post_blood_gmm.items():
    rows = export_obs.index[export_obs.library_id.eq(library)]
    assert raw_by_library[library].obs.loc[rows, "qc_pass"].all()
    posterior = model.predict_proba(export_obs.loc[rows, ["log_counts", "log_genes"]].to_numpy())
    export_obs.loc[rows, "qc_high_complexity_probability"] = posterior[:, np.argmax(model.means_[:, 0])]
    export_obs.loc[rows, "qc_complexity_confidence"] = posterior.max(axis=1)
export_var = qc_full.var.loc[export_genes].copy()
source_files, reference_features = ({}, None)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


for library, prefix in SOURCES.items():
    source_files[library] = {}
    for kind, suffix in (("matrix", "mtx.gz"), ("features", "tsv.gz"), ("barcodes", "tsv.gz")):
        # Set the data file or directory path.
        path = ROOT
        source_files[library][kind] = {"path": str(path), "sha256": file_sha256(path)}
    # Set the data file or directory path.
    features = pd.read_csv(ROOT, sep="\t", header=None, dtype=str)
    if reference_features is None:
        reference_features = features
    else:
        pd.testing.assert_frame_equal(features, reference_features)
source_rows = pd.DataFrame(
    {
        "gene_symbol": reference_features[1],
        "source_feature_id": reference_features[0],
        "feature_row_id": (np.arange(len(reference_features)) + 1).astype(str),
    }
)
gene_sources = source_rows.groupby("gene_symbol", sort=False).agg(";".join).loc[export_genes]
assert gene_sources.source_feature_id.equals(export_var.source_gene_ids.rename("source_feature_id"))
assert reference_features[0].str.fullmatch("ENSMUSG\\d+(?:\\.\\d+)?").all()
assert reference_features[2].eq("Gene Expression").all()
for key in gene_sources:
    export_var[key] = gene_sources[key]
export_var["gene_symbol"] = export_genes.astype(str)
export_var["gene_id_original"] = export_var.source_feature_id
export_var["gene_id_analysis"] = export_genes.astype(str)
export_var["gene_id_type"] = "gene_symbol"
export_var["source_gene_id_type"] = "Ensembl_gene_id"
merged = export_var.source_feature_id.str.contains(";", regex=False)
export_var["gene_id_repair"] = np.where(merged, "duplicate_symbols_summed", "none")
export_var["canonical_gene_id"] = export_var.source_feature_id.str.replace("\\.\\d+$", "", regex=True).mask(
    merged, ""
)
export_var["canonical_mapping_status"] = np.where(
    merged, "multiple_source_ids_unresolved", "source_ensembl_candidate_unharmonized"
)
export_var["integration_gene_id_candidate"] = export_var.canonical_gene_id
export_var["eligible_for_within_dataset_analysis"] = True
export_var["eligible_for_cross_dataset_integration"] = False
export_var["cross_dataset_feature_blocker"] = "versioned_cross_dataset_gene_harmonization_not_run"
export_var["mt_feature"] = export_var.mt
export_var["ribosomal_feature"] = export_var.ribo
export_var["hemoglobin_feature"] = export_var.hb
export_var["ERCC_feature"] = export_genes.str.match("(?i)^ercc[-_]\\d+$")
export_var["reporter_or_vector_feature"] = export_genes.str.match("(?i)^(e?gfp\\d*|tdtomato|mcherry|wpre)$")
export_var["technical_feature"] = export_var.technical
export_var["feature_type"] = "Gene Expression"
export_var["use_for_model_input"] = True
export_var["n_cells_post_stage_qc"] = mother_counts.getnnz(axis=0)[feature_keep]
export_var["total_counts_post_stage_qc"] = np.asarray(mother_counts.sum(axis=0)).ravel()[feature_keep]
export_var["n_cells_in_export"] = export_X.getnnz(axis=0)
export_var["total_counts_in_export"] = np.asarray(export_X.sum(axis=0)).ravel()
export_var["detected_in_exported_cells"] = export_var.n_cells_in_export.gt(0)
for frame in (export_obs, export_var):
    for key in frame.select_dtypes(include=["object", "string"]).columns:
        frame[key] = frame[key].fillna("Unknown").astype(str)
p28_for_integration = ad.AnnData(X=export_X, obs=export_obs, var=export_var)
p28_for_integration.layers["counts"] = export_X.copy()
# Set the notebook file path.
notebook_path = ROOT


def json_text(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=lambda x: x.item())


p28_for_integration.uns["matrix_contract"] = {
    "schema_version": "1.0.0",
    "X_semantics": "raw_integer_UMI_counts",
    "counts_layer_semantics": "exact_copy_of_X",
    "sparse_format": "CSR",
    "dtype": "int32",
    "normalization_applied": False,
    "log1p_applied": False,
    "scaling_applied": False,
    "raw_present": False,
}
p28_for_integration.uns["selection_contract"] = {
    "purpose": "P28 SGN stagewise input for later integration",
    "mother_object": "qc_full after residual immune removal",
    "feature_axis": "global_expression.var_names after stage QC; not local subset or HVGs",
    "minimum_cells_in_qc_mother": int(MIN_GENE_CELLS),
    "selection_chain": "#3 global SGN region -> #4 joint QC -> #5 neural/glial screen -> #6 marker-supported cells -> #7 reviewed HVG2000 annotation",
    "source_notebook": str(notebook_path),
    "final_n_cells": p28_for_integration.n_obs,
    "global_clusters": sorted(target_region_raw.obs.global_cluster.astype(str).unique().tolist()),
    "seed_cluster_map": P28_CLUSTER_MAP[2000],
    "final_cluster_map": FINAL_CLUSTER_MAP.copy(),
    "target_qc_rule": f"exclude n_genes < {LOCAL_LOW_GENES} AND pct_mt > {LOCAL_MT_MAX}",
    "marker_screen": p28_clean_raw.uns["cell_type_screen"].copy(),
    "qc_summary_json": json_text(qc_summary.to_dict(orient="index")),
    "initial_QC_exclusions_json": json_text(
        {lib: a.obs.exclusion_reason.value_counts().to_dict() for lib, a in raw_by_library.items()}
    ),
    "subtype_selection_counts_json": json_text(
        pd.crosstab(p28_final_selection.p28_selection_reason, p28_final_selection.library_id).to_dict()
    ),
    "marker_review_counts_json": json_text(
        pd.crosstab(p28_cell_audit.decision, p28_cell_audit.library_id).to_dict()
    ),
    "residual_immune_removed": len(immune_ids),
    "annotation_status": "final_cluster_annotation",
    "annotation_disagreements_retained": int(disagreement.sum()),
    "no_additional_cell_filter_at_export": True,
}
p28_for_integration.uns["integration_contract"] = {
    "counts_layer": "counts",
    "batch_key": "integration_batch",
    "labels_key": "scanvi_label",
    "unlabeled_category": "Unknown",
    "gene_harmonization_status": "not_run",
    "cross_dataset_blocker": "versioned_cross_dataset_gene_harmonization_not_run",
    "HVG_comparison_targets": np.array([2000, 3000, 5000]),
    "scanvi_max_epochs": 400,
    "batch_stage_confounded": True,
    "label_policy": "Keep all final cell_type annotations; annotation_disagreement cells are unlabeled for scANVI",
    "next_input_steps": "Versioned gene harmonization, common gene axis, concat, joint batch-aware HVG comparison",
    "harmony_input": "Normalize/log1p/scale/PCA on a separate work copy after harmonization and concat",
}
p28_for_integration.uns["source_provenance"] = {
    "dataset_id": DATASET_ID,
    "study_id": DATASET_ID,
    "libraries": list(SOURCES),
    "raw_files": source_files,
    "source_notebook": str(notebook_path),
    "notebook_sha256_at_export": file_sha256(notebook_path),
    "export_time_utc": datetime.now(timezone.utc).isoformat(),
    "code_version": "P28_clustering_export_v1",
    "matrix_relationship": "Raw source feature rows summed by exact gene symbol in #1; selected cells recovered from qc_full",
    "source_duplicate_symbol_rows": int(reference_features[1].duplicated().sum()),
    "feature_row_id_definition": "Semicolon-separated 1-based rows in identical source features tables",
    "gene_id_type_definition": "gene_id_analysis uses symbols; gene_id_original/source_feature_id preserve Ensembl source IDs",
    "missing_QC_field": "n_features_source is NaN: pre-symbol-merge detected feature counts were not retained; n_genes_source counts merged symbols",
    "QC_thresholds": {
        "mito_max_pct": MT_MAX_PCT,
        "hb_exclude_ge_pct": HB_MAX_PCT,
        "doublet_ge": DOUBLET_THRESHOLDS,
    },
    "complexity_fields": "Posterior from the existing post-blood GMM; confidence is max posterior, not an independent quality probability",
    "software_versions": {
        p: importlib.metadata.version(p) for p in ("anndata", "scanpy", "numpy", "pandas", "scipy")
    },
}


def ordered_id_sha256(values):
    encoded = json.dumps(list(map(str, values)), ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def csr_sha256(matrix):
    h = hashlib.sha256()
    for values in (matrix.shape, matrix.indptr, matrix.indices, matrix.data):
        h.update(np.asarray(values, dtype="<i8").tobytes())
    return h.hexdigest()


expected_fingerprints = {
    "ordered_cell_id_sha256": ordered_id_sha256(export_ids),
    "ordered_feature_id_sha256": ordered_id_sha256(export_genes),
    "CSR_matrix_sha256": csr_sha256(export_X),
    "nnz": int(export_X.nnz),
    "total_counts": int(export_X.sum()),
}
p28_for_integration.uns["export_contract"] = {
    **expected_fingerprints,
    "output_filename": export_path.name,
    "n_obs": p28_for_integration.n_obs,
    "n_vars": p28_for_integration.n_vars,
    "feature_exclusions": "mt, Hba/Hbb, ERCC, reporter/vector, technical; detection < MIN_GENE_CELLS in the QC mother",
    "zero_genes_in_export_retained": True,
    "saved_slots": ["X", "layers/counts", "obs", "var", "uns/contracts"],
    "omitted_slots": ["raw", "obsm", "obsp", "varm", "varp"],
    "normalized_HVG_integration_embeddings_saved": False,
    "fingerprint_encoding": "IDs: UTF-8 compact JSON array; CSR: shape, indptr, indices, data as little-endian int64 bytes",
}


def validate_p28_export(a):
    assert a.obs_names.equals(export_ids)
    assert a.var_names.equals(export_genes)
    assert a.obs_names.is_unique and a.var_names.is_unique
    assert a.var_names.str.casefold().is_unique
    assert np.array_equal(a.obs.cell_id_global.astype(str).to_numpy(), a.obs_names.to_numpy())
    assert a.raw is None and set(a.layers) == {"counts"}
    assert not any((len(slot) for slot in (a.obsm, a.obsp, a.varm, a.varp)))
    for matrix in (a.X, a.layers["counts"]):
        assert sparse.isspmatrix_csr(matrix)
        assert matrix.dtype == np.dtype("int32")
        assert matrix.has_canonical_format
        assert np.isfinite(matrix.data).all()
        assert (matrix.data > 0).all()
    assert all(
        (
            np.array_equal(getattr(a.X, k), getattr(a.layers["counts"], k))
            for k in ("indptr", "indices", "data")
        )
    )
    for key in (
        "selected_for_integration",
        "technical_qc_pass",
        "blood_qc_pass",
        "cell_calling_qc_pass",
        "doublet_qc_pass",
        "target_qc_pass",
        "final_candidate_keep",
    ):
        assert a.obs[key].notna().all() and a.obs[key].all(), key
    assert not a.obs.predicted_doublet.any()
    assert (
        not a.var[
            [
                "mt_feature",
                "hemoglobin_feature",
                "ERCC_feature",
                "reporter_or_vector_feature",
                "technical_feature",
            ]
        ]
        .to_numpy()
        .any()
    )
    assert not a.var.eligible_for_cross_dataset_integration.any()
    assert a.obs.integration_batch.astype(str).equals(batch.rename("integration_batch"))
    assert a.obs.cell_type.astype(str).equals(final_labels)
    assert np.array_equal(
        a.obs.scanvi_label.astype(str).to_numpy(), final_labels.mask(disagreement, "Unknown").to_numpy()
    )
    assert a.obs.scanvi_supervision_status.notna().all()
    assert np.array_equal(a.obs.total_counts_analysis.to_numpy(), np.asarray(a.X.sum(axis=1)).ravel())
    assert np.array_equal(a.obs.n_genes_analysis.to_numpy(), a.X.getnnz(axis=1))
    assert np.array_equal(a.var.n_cells_in_export.to_numpy(), a.X.getnnz(axis=0))
    assert np.array_equal(a.var.total_counts_in_export.to_numpy(), np.asarray(a.X.sum(axis=0)).ravel())
    assert np.array_equal(a.var.detected_in_exported_cells.to_numpy(), a.X.getnnz(axis=0) > 0)
    actual = {
        "ordered_cell_id_sha256": ordered_id_sha256(a.obs_names),
        "ordered_feature_id_sha256": ordered_id_sha256(a.var_names),
        "CSR_matrix_sha256": csr_sha256(a.X),
        "nnz": int(a.X.nnz),
        "total_counts": int(a.X.sum()),
    }
    assert actual == expected_fingerprints
    assert all((a.uns["export_contract"][k] == v for k, v in actual.items()))


p28_for_integration.uns["source_provenance"][
    "original_analysis_notebook"
] = 'fig1_P28_clustering.ipynb'
p28_for_integration.uns["source_provenance"][
    "retained_HVG_branches"
] = "immune union: 2000/3000/5000; global: 2000; local: 2000; candidate: 3000; final: 2000"
p28_for_integration.uns["source_provenance"][
    "omitted_exploratory_obs"
] = "three_branch_agreement; final/seed cluster columns for unused 3000/5000 branches"
validate_p28_export(p28_for_integration)
if export_path.exists():
    checked = ad.read_h5ad(export_path)
    validate_p28_export(checked)
    for key in (
        "library_id",
        "cell_type",
        "candidate_type",
        "annotation_disagreement",
        "scanvi_label",
        "scanvi_supervision_status",
    ):
        assert np.array_equal(
            checked.obs[key].astype(str).to_numpy(), p28_for_integration.obs[key].astype(str).to_numpy()
        ), key
    for key in (
        "source_feature_id",
        "gene_id_original",
        "gene_id_analysis",
        "canonical_gene_id",
        "canonical_mapping_status",
    ):
        assert np.array_equal(
            checked.var[key].astype(str).to_numpy(), p28_for_integration.var[key].astype(str).to_numpy()
        ), key
    del checked
else:
    export_path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix="P28_export_", suffix=".tmp.h5ad", dir=export_path.parent)
    os.close(fd)
    temporary_path = Path(temporary_name)
    try:
        p28_for_integration.write_h5ad(
            temporary_path, compression="gzip", convert_strings_to_categoricals=False
        )
        checked = ad.read_h5ad(temporary_path)
        validate_p28_export(checked)
        pd.testing.assert_frame_equal(checked.obs, p28_for_integration.obs)
        pd.testing.assert_frame_equal(checked.var, p28_for_integration.var)
        del checked
        if export_path.exists():
            raise FileExistsError(f"Existing file was not overwritten: {export_path}")
        os.rename(temporary_path, export_path)
        checked = ad.read_h5ad(export_path)
        validate_p28_export(checked)
        del checked
    finally:
        if temporary_path.exists():
            temporary_path.unlink()
runtime.cleanup()
if not any(cache_root.iterdir()):
    cache_root.rmdir()


In [ ]:
#9. =========== Migrate the P28 integration input to schema 2.0.0
from copy import deepcopy

# Set the output directory path.
output_dir = ROOT
# Set the stage result path; uses {stage}_cluster_intergration.h5ad generated by P1, P7 or P28 clustering.
output_path = ROOT
# Set the result file path; uses P28_GSE202920__GSM6138444-GSM6138445__SGN_for_integration.h5ad generated by P28 clustering export.
legacy_path = ROOT

a = p28_for_integration.copy()
previous = deepcopy(dict(a.uns))
before_obs, before_var = a.obs.copy(), a.var.copy()
blocker = "versioned_cross_dataset_gene_harmonization_not_run"
types = ["Type Ia", "Type Ib", "Type Ic", "Type II"]

assert a.obs_names.equals(p28_final.obs_names)
assert a.var_names.equals(global_expression.var_names)
assert a.obs.cell_type.astype(str).equals(
    p28_final.obs.cell_type.astype(str)
)
assert a.obs_names.is_unique and a.var_names.str.casefold().is_unique
assert previous["integration_contract"]["gene_harmonization_status"] == "not_run"
assert previous["matrix_contract"]["X_semantics"] == "raw_integer_UMI_counts"
assert sparse.isspmatrix_csr(a.X)
assert sparse.isspmatrix_csr(a.layers["counts"])
assert (a.X != a.layers["counts"]).nnz == 0
assert (
    a.layers["counts"]
    != qc_full[a.obs_names, a.var_names].layers["counts"]
).nnz == 0


def fingerprints_v2(obj):
    return {
        "ordered_cell_id_sha256": ordered_id_sha256(obj.obs_names),
        "ordered_feature_id_sha256": ordered_id_sha256(obj.var_names),
        "X_sha256": csr_sha256(obj.X),
        "counts_sha256": csr_sha256(obj.layers["counts"]),
        "X_nnz": int(obj.X.nnz),
        "counts_nnz": int(obj.layers["counts"].nnz),
        "total_counts": int(obj.layers["counts"].sum()),
    }


expected = fingerprints_v2(a)
legacy_sha256 = (
    file_sha256(legacy_path) if legacy_path.exists()
    else previous["export_contract"].get("superseded_file_sha256", "")
)

# Preserve the previous annotation policy as history.
for key in ("scanvi_label", "annotation_status", "scanvi_supervision_status"):
    history_key = key + "_before_schema_v2"
    if history_key not in a.obs:
        a.obs[history_key] = a.obs[key].copy()

assert a.obs.cell_type.notna().all()
assert a.obs.cell_type.astype(str).isin(types).all()

a.obs["cell_type"] = pd.Categorical(
    a.obs.cell_type.astype(str), categories=types, ordered=True
)
a.obs["scanvi_label"] = a.obs.cell_type.copy()
a.obs["annotation_status"] = "final_stagewise_annotation"
a.obs["scanvi_supervision_status"] = "labeled_final_stagewise_annotation"
a.obs["selected_for_integration"] = True
a.obs["stage_analysis"] = a.obs.stage_standardized
a.obs["batch_id_global"] = a.obs.integration_batch

for key in ("predicted_doublet", "doublet_qc_pass"):
    a.obs[key] = a.obs[key].astype("boolean")

assert a.obs[["predicted_doublet", "doublet_qc_pass"]].notna().all().all()
assert not a.obs.predicted_doublet.any() and a.obs.doublet_qc_pass.all()

a.obs["doublet_status_before_schema_v2"] = (
    a.obs["doublet_status_before_schema_v2"]
    if "doublet_status_before_schema_v2" in a.obs
    else a.obs.doublet_status.copy()
)
a.obs["doublet_status"] = "assessed_below_library_threshold"

# Preserve original IDs before separating candidates from canonical IDs.
for key in (
    "canonical_gene_id", "canonical_mapping_status",
    "integration_gene_id_candidate", "source_gene_id_type",
):
    history_key = key + "_before_schema_v2"
    if history_key not in a.var:
        a.var[history_key] = a.var[key].copy()

source_ids = a.var.source_feature_id.astype(str)
assert source_ids.str.split(";").explode().str.fullmatch(
    r"ENSMUSG\d+(?:\.\d+)?"
).all()

multiple_ids = source_ids.str.contains(";", regex=False)
a.var["gene_symbol"] = a.var_names.astype(str)
a.var["gene_id_analysis"] = a.var_names.astype(str)
a.var["gene_id_type"] = "gene_symbol"
a.var["source_gene_id_type"] = np.where(
    multiple_ids, "ensembl_gene_id_list", "ensembl_gene_id"
)
a.var["integration_gene_id_candidate"] = (
    source_ids.str.replace(r"\.\d+$", "", regex=True)
    .mask(multiple_ids, "")
)
a.var["integration_gene_id_candidate_type"] = np.where(
    multiple_ids, "unresolved", "ensembl_gene_id"
)
a.var["canonical_gene_id"] = ""
a.var["canonical_mapping_status"] = np.where(
    multiple_ids, "multiple_source_ids_unresolved", "not_run"
)
a.var["eligible_for_cross_dataset_integration"] = False
a.var["cross_dataset_feature_blocker"] = blocker

if "source_ribosomal_feature" not in a.var:
    a.var["source_ribosomal_feature"] = (
        a.var.ribosomal_feature.astype("boolean")
    )
a.var["ribosomal_feature"] = pd.array(
    [pd.NA] * a.n_vars, dtype="boolean"
)
a.var["ribosomal_classification_status"] = "pending_shared_gene_set"

counts = a.layers["counts"]
a.obs["total_counts_analysis"] = np.asarray(counts.sum(axis=1)).ravel()
a.obs["n_genes_analysis"] = np.asarray((counts > 0).sum(axis=1)).ravel()
a.var["n_cells_in_export"] = np.asarray((counts > 0).sum(axis=0)).ravel()
a.var["total_counts_in_export"] = np.asarray(counts.sum(axis=0)).ravel()
a.var["detected_in_exported_cells"] = a.var.total_counts_in_export.gt(0)


def contracts_json(value):
    return json.dumps(
        value, ensure_ascii=False, sort_keys=True,
        default=lambda x: x.tolist() if isinstance(x, np.ndarray) else x.item(),
    )


# Keep previous export contracts as provenance.
provenance = deepcopy(previous["source_provenance"])
if previous["matrix_contract"]["schema_version"] != "2.0.0":
    provenance["migration_history"] = {
        "from_schema": previous["matrix_contract"]["schema_version"],
        "to_schema": "2.0.0",
        "previous_contracts_json": contracts_json(previous),
        "changes": [
            "All final cell types are supervised; historical disagreement is retained",
            "Unverified canonical IDs and mapping descriptions are preserved as history",
            "Candidate/source IDs remain available; cross-dataset mapping is not run",
            "Shared ribosomal classification is pending; source classification is retained",
            "Previous HVG and training plans are historical, not active save requirements",
        ],
    }

source_files, source_hashes = {}, {}
for library, files in provenance["raw_files"].items():
    for role, info in files.items():
        path = Path(info["path"])
        actual_hash = file_sha256(path)
        assert actual_hash == info["sha256"], f"Source file changed: {path}"
        key = f"{library}__{role}"
        source_files[key] = {
            "library_id": library,
            "role": role,
            "path": str(path),
        }
        source_hashes[key] = actual_hash

# Set the notebook file path.
provenance.update({
    "schema_version": "2.0.0",
    "source_files": source_files,
    "source_file_sha256": source_hashes,
    "source_dataset": str(a.obs.dataset_id.iloc[0]),
    "source_library": list(a.obs.library_id.astype(str).unique()),
    "source_object": "p28_for_integration; counts verified against qc_full",
    "source_reference_assembly": "not_verified",
    "source_annotation_release": "not_verified",
    "source_reference_evidence": "No versioned reference provenance verified",
    "export_notebook": str(ROOT),
    "export_date": datetime.now(timezone.utc).isoformat(),
    "export_time_utc": datetime.now(timezone.utc).isoformat(),
    "code_version": "P28_schema_v2_migration",
    "export_notebook_sha256": file_sha256(ROOT),
    "software_versions": {
        p: importlib.metadata.version(p)
        for p in ("anndata", "scanpy", "numpy", "pandas", "scipy")
    },
    "qc_metric_definitions": {
        "total_counts_analysis": "Sum of verified counts on the retained export axis",
        "n_genes_analysis": "Number of genes with counts > 0 on the export axis",
        "source_counts": (
            "Full source symbol axis; duplicate source rows were summed by symbol"
        ),
        "endogenous_counts": (
            "Full source symbol axis excluding source technical features"
        ),
        "n_features_source": (
            "NA; per-cell detections before symbol merging were not retained"
        ),
        "pct_mito": "0-100 percent; full source counts, source mt[-_] classification",
        "pct_hb": "0-100 percent; full source counts, source Hba/Hbb classification",
        "pct_ribo": r"0-100 percent; full source counts, source ^Rp[sl]\d classification",
        "pct_reporter": (
            "0-100 percent; full source counts, documented source reporter rules"
        ),
        "doublets": (
            "Per-library Scrublet scores; completed thresholds preserved in QC_thresholds"
        ),
        "shared_ribosomal_classification": "pending_shared_gene_set",
    },
})

selection = deepcopy(previous["selection_contract"])
selection.update({
    "schema_version": "2.0.0",
    "source_object": "p28_final; selected cells recovered on the qc_full analysis axis",
    "selection_rule": selection["selection_chain"],
    "n_selected_cells": a.n_obs,
    "cell_type_counts": {
        t: int(a.obs.cell_type.eq(t).sum()) for t in types
    },
    "annotation_status": "final_stagewise_annotation",
    "annotation_source": "Reviewed final HVG2000 clusters; original mapping retained",
    "label_policy": "all_final_cell_types",
    "annotation_disagreements_retained": int(a.obs.annotation_disagreement.sum()),
})

source_genes = qc_full.var_names
reporter_mask = source_genes.str.match(
    r"(?i)^(e?gfp\d*|tdtomato\d*|mcherry\d*|wpre\d*)$"
)
technical_mask = (
    qc_full.var[["mt", "hb", "technical"]].any(axis=1).to_numpy()
    | source_genes.str.match(r"(?i)^ercc[-_]\d+$")
    | reporter_mask
)
assert not a.var_names.isin(source_genes[technical_mask]).any()

a.uns.clear()
a.uns["matrix_contract"] = {
    "schema_version": "2.0.0",
    "raw_counts_available": True,
    "raw_counts_status": "available_verified",
    "counts_layer": "counts",
    "counts_layer_semantics": "raw_counts",
    "count_type": "UMI",
    "counts_verification_source": (
        "Source 10x MTX hashes verified; exact cell/gene-aligned match to qc_full raw counts"
    ),
    "counts_dtype": str(counts.dtype),
    "counts_sparse_format": "CSR",
    "X_semantics": "raw_integer_UMI_counts",
    "X_dtype": str(a.X.dtype),
    "X_sparse_format": "CSR",
    "X_normalization_status": "not_applied",
    "X_log1p_status": "not_applied",
    "X_scaling_status": "not_applied",
    "X_counts_relationship": "exact_copy",
    "raw_present": False,
}
a.uns["selection_contract"] = selection
a.uns["integration_contract"] = {
    "schema_name": "auditory_neuron_stagewise",
    "schema_version": "2.0.0",
    "input_role": "stagewise_pre_harmonization",
    "counts_layer": "counts",
    "batch_key": "integration_batch",
    "labels_key": "scanvi_label",
    "unlabeled_category": "Unknown",
    "label_policy": "all_final_cell_types",
    "gene_harmonization_status": "not_run",
    "cross_dataset_blocker": blocker,
    "stage_as_batch": False,
    "batch_stage_confounded": True,
    "count_model_compatible": True,
    "model_ready": False,
    "model_blockers": [blocker],
}
a.uns["source_provenance"] = provenance
a.uns["export_contract"] = {
    **expected,
    "schema_version": "2.0.0",
    "output_filename": output_path.name,
    "n_obs": a.n_obs,
    "n_vars": a.n_vars,
    "saved_slots": ["X", "layers/counts", "obs", "var", "uns/contracts"],
    "excluded_feature_classes": [
        "mitochondrial", "hemoglobin", "ERCC", "reporter/vector", "technical"
    ],
    "excluded_feature_ids": source_genes[technical_mask].tolist(),
    "excluded_feature_ids_scope": "Technical gene symbols excluded from the source axis",
    "feature_axis": "Unchanged stable post-QC axis; no additional subset gene filtering",
    "zero_count_genes_retained": int((~a.var.detected_in_exported_cells).sum()),
    "superseded_file": str(legacy_path),
    "superseded_file_sha256": legacy_sha256,
    "fingerprint_encoding": (
        "SHA256; ordered IDs: UTF-8 compact JSON array; CSR: shape, indptr, indices, "
        "data concatenated as little-endian int64 bytes; X and counts recomputed separately"
    ),
}

expected_contracts = contracts_json(dict(a.uns))


def validate_v2(obj):
    assert obj.obs_names.equals(before_obs.index)
    assert obj.var_names.equals(before_var.index)
    assert np.array_equal(obj.obs.cell_id_global.astype(str), obj.obs_names)
    assert obj.raw is None and set(obj.layers) == {"counts"}
    assert not any(len(s) for s in (obj.obsm, obj.obsp, obj.varm, obj.varp))

    for matrix in (obj.X, obj.layers["counts"]):
        assert sparse.isspmatrix_csr(matrix)
        assert matrix.dtype == np.dtype("int32")
        assert np.isfinite(matrix.data).all() and (matrix.data >= 0).all()

    assert (obj.X != obj.layers["counts"]).nnz == 0
    assert fingerprints_v2(obj) == expected
    assert contracts_json(dict(obj.uns)) == expected_contracts
    assert all(obj.uns["export_contract"][k] == v for k, v in expected.items())

    for key in (
        "cell_type", "candidate_type", "annotation_disagreement",
        "original_cell_id", "source_cell_id", "dataset_id",
        "library_id", "integration_batch",
    ):
        assert obj.obs[key].astype(str).equals(before_obs[key].astype(str)), key

    for key in ("source_feature_id", "gene_id_original", "feature_row_id"):
        assert obj.var[key].astype(str).equals(before_var[key].astype(str)), key

    assert obj.obs.scanvi_label.notna().all()
    assert obj.obs.scanvi_label.astype(str).equals(obj.obs.cell_type.astype(str))
    assert obj.obs.scanvi_label.cat.categories.tolist() == types
    assert obj.obs.annotation_status.eq("final_stagewise_annotation").all()
    assert obj.obs.scanvi_supervision_status.eq(
        "labeled_final_stagewise_annotation"
    ).all()
    assert obj.obs.selected_for_integration.all()

    assert str(obj.obs.predicted_doublet.dtype) == "boolean"
    assert str(obj.obs.doublet_qc_pass.dtype) == "boolean"
    assert obj.obs[["predicted_doublet", "doublet_qc_pass"]].notna().all().all()
    assert not obj.obs.predicted_doublet.any() and obj.obs.doublet_qc_pass.all()

    assert obj.var.canonical_gene_id.eq("").all()
    assert obj.var.ribosomal_feature.isna().all()
    assert not obj.var.eligible_for_cross_dataset_integration.any()
    assert obj.uns["integration_contract"]["model_ready"] == False
    assert obj.uns["matrix_contract"]["raw_counts_available"] == True

    np.testing.assert_array_equal(
        obj.obs.total_counts_analysis,
        np.asarray(obj.layers["counts"].sum(axis=1)).ravel(),
    )
    np.testing.assert_array_equal(
        obj.var.total_counts_in_export,
        np.asarray(obj.layers["counts"].sum(axis=0)).ravel(),
    )


validate_v2(a)
output_dir.mkdir(parents=True, exist_ok=True)
# Set the temporary working directory or file path.
temporary_root = ROOT
created_temporary_root = not temporary_root.exists()
temporary_root.mkdir(parents=True, exist_ok=True)

try:
    with tempfile.TemporaryDirectory(
        prefix="export_", dir=temporary_root
    ) as work:
        temporary_file = Path(work) / output_path.name
        a.write_h5ad(
            temporary_file,
            compression="gzip",
            convert_strings_to_categoricals=False,
        )

        checked = ad.read_h5ad(temporary_file)
        validate_v2(checked)
        pd.testing.assert_frame_equal(checked.obs, a.obs)
        pd.testing.assert_frame_equal(checked.var, a.var)
        del checked

        os.replace(temporary_file, output_path)

        checked = ad.read_h5ad(output_path)
        validate_v2(checked)
        pd.testing.assert_frame_equal(checked.obs, a.obs)
        pd.testing.assert_frame_equal(checked.var, a.var)
        del checked

    if legacy_path.exists():
        assert legacy_path.parent.resolve() == output_path.parent.resolve()
        assert file_sha256(legacy_path) == legacy_sha256, (
            "The legacy file changed during export."
        )
        legacy_path.unlink()

finally:
    if (
        created_temporary_root
        and temporary_root.exists()
        and not any(temporary_root.iterdir())
    ):
        temporary_root.rmdir()

p28_for_integration = a
print(f"Saved and validated: {output_path}")
print(
    f"Shape: {a.shape} | Verified UMI counts | "
    "Unknown labels: 0 | Schema: 2.0.0"
)